# Faster R-CNN — Kaggle smoke online ON (run2)

Notebook thử riêng, giữ nguyên notebook03 scaffold của nhóm.

**Mục tiêu:** chạy 2 epoch trên 2 Tesla T4 để kiểm train với online jitter 0.1. Giữ pretrained COCO, custom head, seed42, batch tổng 8/eval16 và AMP như run OFF. W&B tắt tường minh. Không dùng metric smoke để chọn model hoặc báo cáo chính thức; split cachev1 còn hạn chế frame/cảnh gần nhau.


## 1. Bootstrap source và dependencies

Kiểm commit đúng run OFF. Không tự reset checkout có thay đổi. Cell này có thể tải source/dependency; cell train sẽ tải pretrained nếu cache weights chưa có.


In [ ]:
import os, sys, subprocess, json, time, uuid
from pathlib import Path

REPO = Path('/kaggle/working/VN-Traffic-Sign-Detection')
EXPECTED_SHA = '17ad0eea18df9d414b39748a89eb39d9d06399df'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', 'faster-rcnn',
                    'https://github.com/tuan8p/VN-Traffic-Sign-Detection.git', str(REPO)], check=True)
sha = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
print('Source:', sha)
assert sha == EXPECTED_SHA, 'Source đã thay đổi: đối chiếu commit trước khi chạy tiếp'
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
# Giữ torch/torchvision do Kaggle cung cấp; không upgrade ngầm hoặc cài Pixi win-64 lên Linux.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO),
                'wandb', 'pycocotools'], check=True)


## 2. Kiểm hai GPU, CUDA NMS và tìm processed cache

Nếu fail, dừng và giữ traceback cùng versions. NCCL available mới là kiểm bản build; run DDP thành công mới chứng minh giao tiếp thực tế.


In [ ]:
import torch, torchvision, pycocotools
from torchvision.ops import nms

print('Python:', sys.version)
print('Torch:', torch.__version__, 'TorchVision:', torchvision.__version__)
print('CUDA:', torch.version.cuda, 'GPUs:', torch.cuda.device_count())
assert torch.cuda.is_available() and torch.cuda.device_count() == 2
assert torch.distributed.is_nccl_available(), 'Runtime chưa có NCCL'
assert hasattr(torch.amp, 'GradScaler'), 'Torch runtime chưa phù hợp API AMP pipeline'
for index in range(2):
    print(index, torch.cuda.get_device_name(index))
    device = f'cuda:{index}'
    kept = nms(torch.tensor([[0., 0., 10., 10.]], device=device),
               torch.tensor([.9], device=device), .5)
    assert kept.cpu().tolist() == [0]
    torch.cuda.synchronize(index)

candidates = sorted(p.parent for p in Path('/kaggle/input').rglob('data.yaml')
                    if p.parent.name == 'processed')
print('Processed roots:', candidates)
assert len(candidates) == 1, 'Chọn đúng một cache processed; không fallback sang raw/local'
DATA_ROOT = candidates[0]


## 3. Kiểm toàn bộ cache

Decode/hash ảnh và nhãn; có thể mất khoảng14phút theo run OFF trước đó. Không chỉnh dữ liệu. Các kiểm IO không chứng minh độc lập cảnh/video.


In [ ]:
from vn_tsd.config.resolve import resolve_config
from vn_tsd.pipelines.faster_rcnn.config import validate_config
from vn_tsd.pipelines.faster_rcnn.dataset import load_cache

cfg = validate_config(resolve_config(
    pipeline_yaml=REPO/'configs/pipelines/faster_rcnn.yaml',
    shared_yaml=REPO/'configs/shared.yaml', runtime_yaml=REPO/'configs/runtime/kaggle.yaml',
    overrides={'data': {'root': str(DATA_ROOT)}, 'train': {'require_wandb': False}}))
sets, identity = load_cache(cfg)
meta = json.loads((DATA_ROOT/'meta.json').read_text(encoding='utf-8'))
expected = {'train': (2660, 5743), 'val': (438, 1204), 'test': (371, 829)}
stems = {}
for split, dataset in sets.items():
    info = dataset.identity
    print(split, 'images', info['count'], 'boxes', info['annotations'],
          'small_boxes', info['small_boxes'], 'hash', info['sha256'])
    assert (info['count'], info['annotations']) == expected[split], 'Không khớp snapshotv1'
    assert info['count'] == meta['splits'][split]['n_images']
    assert info['annotations'] == meta['splits'][split]['n_boxes_out']
    listed = [Path(line.strip().replace('\\', '/')).stem for line in
              (DATA_ROOT/f'{split}_files.txt').read_text(encoding='utf-8').splitlines() if line.strip()]
    stems[split] = {r['path'].stem for r in dataset.records}
    assert len(listed) == len(set(listed)) and set(listed) == stems[split]
    assert info['small_boxes'] == 0, 'Báo Bình về box dưới policy8px; không tự lọc'
for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    assert not stems[a] & stems[b], f'Trùng stem: {a}/{b}'
for split in ('val', 'test'):
    assert not any('_aug_' in stem for stem in stems[split])
    for stem in stems['train']:
        if '_aug_' in stem:
            assert stem.split('_aug_')[0] not in stems[split], 'Aug parent ở split khác'
seen = {label for r in sets['train'].records for label in r['labels']}
assert seen == set(range(1, 53)), 'Train thiếu lớp'
print('Full cache adapter check passed; chưa chứng minh độc lập cảnh/near-duplicate')


## 4. Tạo run online ON mới

So sánh hash từng split với run OFF; dừng nếu cache khác. Mỗi lần chạy tạo tên/thư mục mới.


In [ ]:
import yaml

expected_hashes = {
    'train': 'fd15fcbf42da08bf274c99b267f1cae512463162a6f0b28ed84d31941c0dbd23',
    'val': 'e5139f467e918e47da0bad10c0068e5bcef193c45ab038bebd0007a475e6d9b3',
    'test': '3e02b6face375fcbcf441455970e515a4c5350e1638bd76c01ee6d7d62c6d7bb',
}
actual_hashes = {split: info['sha256'] for split, info in identity['splits'].items()}
print('Dataset split hashes:', actual_hashes)
assert actual_hashes == expected_hashes, 'Cache khác run OFF: dừng, đối chiếu version trước khi chạy'
assert sha == EXPECTED_SHA, 'Source khác run OFF: đối chiếu trước khi chạy'

RUN_NAME = 'frcnn-preflight-online-' + uuid.uuid4().hex[:8]
RUN_DIR = Path('/kaggle/working/frcnn_preflight') / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=False)
OVERRIDES = RUN_DIR/'launch_overrides.yaml'
overrides = {
    'train': {'epochs': 2, 'require_wandb': False},
    'online_preprocess': {'enabled': True},
    'model': {'pretrained': True},
    'eval': {'auto_eval_test': True},
}
OVERRIDES.write_text(yaml.safe_dump(overrides), encoding='utf-8')
(RUN_DIR/'cache_preflight.json').write_text(
    json.dumps({'source_sha': sha, 'kaggle_dataset':
        'zero001206/vn-traffic-sign-detection-preprocessed-data', 'version': 1,
        'scope': 'technical smoke online ON; W&B explicitly disabled', 'dataset': identity},
        ensure_ascii=False, indent=2), encoding='utf-8')
with (RUN_DIR/'pip-freeze.txt').open('w') as versions:
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=versions, check=True)
print('Run mới:', RUN_DIR)


## 5. Xác nhận config trước khi dùng GPU train

Chỉ jitter hình học được bật trong preset này. Không resize/normalize hai lần, không hflip.


In [ ]:
from vn_tsd.pipelines.faster_rcnn.transforms import describe_effective
from vn_tsd.utils.io import load_yaml

launch_overrides = load_yaml(OVERRIDES)
launch_overrides['data'] = {'root': str(DATA_ROOT)}
launch_cfg = validate_config(resolve_config(
    pipeline_yaml=REPO/'configs/pipelines/faster_rcnn.yaml',
    shared_yaml=REPO/'configs/shared.yaml', runtime_yaml=REPO/'configs/runtime/kaggle.yaml',
    overrides=launch_overrides))
online = describe_effective(launch_cfg)['effective']
assert online['enabled'] is True and online['jitter'] == 0.1
assert all(online[k] is False for k in ('hflip', 'color_jitter', 'mosaic', 'mixup'))
assert launch_cfg['seed'] == 42
assert launch_cfg['train']['epochs'] == 2 and launch_cfg['train']['batch_size'] == 8
assert launch_cfg['eval']['batch_size'] == 16 and launch_cfg['train']['require_wandb'] is False
assert launch_cfg['model']['head'] == 'custom_mlp' and launch_cfg['model']['pretrained'] is True
assert launch_cfg['train']['amp'] and launch_cfg['runtime_cfg']['mixed_precision']
print('Config validated. Online effective:', online)
print('Run directory:', RUN_DIR)


## 6. Train 2 epoch bằng DDP

Cell này thực sự dùng GPU. Giữ log nếu OOM/NCCL/worker error; tìm traceback worker đầu tiên. Không chạy thêm launcher song song. Để xuất log khi run lỗi, có thể bỏ qua cell nghiệm thu và chạy cell đóng gói cuối.


In [ ]:
command = [sys.executable, '-m', 'torch.distributed.run', '--standalone', '--nproc_per_node=2',
    '-m', 'tools.run_train', '--pipeline', 'faster_rcnn',
    '--config', str(REPO/'configs/pipelines/faster_rcnn.yaml'),
    '--shared', str(REPO/'configs/shared.yaml'),
    '--runtime', str(REPO/'configs/runtime/kaggle.yaml'),
    '--data-root', str(DATA_ROOT), '--overrides', str(OVERRIDES),
    '--run-name', RUN_NAME, '--run-dir', str(RUN_DIR), '--online']
(RUN_DIR/'launch_command.json').write_text(json.dumps(command, indent=2), encoding='utf-8')
started = time.monotonic()
# Tee stdout+stderr để xem trực tiếp và giữ lỗi rank gốc cho buổi họp.
with (RUN_DIR/'console.log').open('w', encoding='utf-8') as log:
    process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='')
        log.write(line)
        log.flush()
    exit_code = process.wait()
wall_seconds = time.monotonic() - started
(RUN_DIR/'launch_result.json').write_text(
    json.dumps({'exit_code': exit_code, 'wall_seconds': wall_seconds}), encoding='utf-8')
assert exit_code == 0, f'Giữ console.log và traceback đầu tiên; exit={exit_code}'


## 7. Kiểm run hoàn thành và trạng thái online thực tế

Mục tiêu là loss hữu hạn, update optimizer, DDP và artifacts hợp lệ. Chưa kết luận chất lượng hoặc so ON/OFF từ2epoch.


In [ ]:
history = json.loads((RUN_DIR/'history.json').read_text())
metrics = json.loads((RUN_DIR/'metrics.json').read_text())
assert metrics['status'] == 'completed' and len(history) == 2
assert metrics['effective']['world_size'] == 2 and metrics['effective']['amp'] is True
assert metrics['test_status'] == 'completed'
for path in ['resolved_config.yaml', 'test_metrics.json', 'checkpoints/checkpoint_best.pt']:
    assert (RUN_DIR/path).is_file(), path
print(json.dumps(metrics, ensure_ascii=False, indent=2))
print('Epoch seconds:', [row['seconds'] for row in history])
overhead = max(0., wall_seconds - sum(row['seconds'] for row in history))
estimate_seconds = overhead + 20 * history[-1]['seconds']
print('Ước lượng thô run20epoch (chưa early-stop):', round(estimate_seconds/60, 1), 'phút')
print('Cần đo lại sau vài epoch ổn định; first-run download/cache và artifacts có thể khác.')
print('Giữ toàn bộ run folder, console.log, metrics, history, config và checkpoint; xem figures/.')
assert (RUN_DIR/'run.zip').is_file()
print('ZIP:', RUN_DIR/'run.zip')

effective_online = metrics['effective']['online']['effective']
print('Online effective:', effective_online)
assert effective_online['enabled'] is True
assert effective_online['jitter'] == 0.1
assert all(effective_online[k] is False for k in ('hflip', 'color_jitter', 'mosaic', 'mixup'))
print('ONLINE ON / jitter đã được ghi trong metrics; tiếp tục xem loss, log và artifacts.')

import math
assert metrics['dataset_split_hashes'] == expected_hashes
for row in history:
    assert math.isfinite(row['train_loss']) and math.isfinite(row['val_loss'])
    assert row['successful_updates'] > 0
    assert all(math.isfinite(value) for key in ('train_components', 'val_components')
               for value in row[key].values())
print('Technical smoke checks passed. Đây chưa phải kết quả thực nghiệm chính thức.')


## 8. Đóng gói để tải về

Export toàn bộ run folder gồm log và launch_result sau khi launcher kết thúc. Không lồng thêm run.zip để tránh nhân đôi checkpoint. Bản run.zip gốc vẫn nằm trong RUN_DIR. Cell này cũng dùng được sau khi train fail nếu RUN_DIR đã tạo.

Tải file `*-artifacts.zip` và `.sha256` trong Output; Save Version để giữ outputs. Local lưu vào `.agents/artifacts/<RUN_NAME>/`, rồi gửi đường dẫn để review.


In [ ]:
from zipfile import ZipFile, ZIP_STORED
import hashlib

assert RUN_DIR.is_dir(), 'Chưa tạo run directory'
BUNDLE = Path('/kaggle/working') / f'{RUN_NAME}-artifacts.zip'
assert not BUNDLE.exists(), f'Bundle đã tồn tại: {BUNDLE}'
files = sorted(path for path in RUN_DIR.rglob('*') if path.is_file() and path.name != 'run.zip')
assert files, 'Run chưa có file để export'
with ZipFile(BUNDLE, 'x', compression=ZIP_STORED, allowZip64=True) as archive:
    for path in files:
        archive.write(path, arcname=f'{RUN_NAME}/{path.relative_to(RUN_DIR).as_posix()}')
with ZipFile(BUNDLE) as archive:
    assert archive.testzip() is None, 'Bundle CRC error'
sha256 = hashlib.sha256()
with BUNDLE.open('rb') as stream:
    for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
        sha256.update(block)
checksum_path = BUNDLE.with_suffix(BUNDLE.suffix + '.sha256')
checksum_path.write_text(f'{sha256.hexdigest()}  {BUNDLE.name}\n', encoding='utf-8')
print('Download:', BUNDLE)
print('Checksum:', checksum_path)
print('Local folder: .agents/artifacts/' + RUN_NAME + '/')
